In [ ]:
-- ============================================================
-- mart_trial_goals.sql
-- Mart layer: tracks whether each org completed each trial goal
-- One row per organisation
-- ============================================================

WITH stg AS (
    SELECT * FROM {{ ref('stg_events') }}
    -- Note: replace {{ ref('stg_events') }} with stg_events
    -- if not using dbt
),

-- Count how many times each org did each activity
activity_counts AS (
    SELECT
        ORGANIZATION_ID,
        is_converted,

        -- Goal 1: Created at least 3 shifts (core scheduling behaviour)
        SUM(CASE WHEN ACTIVITY_NAME = 'Scheduling.Shift.Created' 
            THEN 1 ELSE 0 END)                        AS shift_created_count,

        -- Goal 2: Approved at least 1 shift (completed workflow)
        SUM(CASE WHEN ACTIVITY_NAME = 'Scheduling.Shift.Approved' 
            THEN 1 ELSE 0 END)                        AS shift_approved_count,

        -- Goal 3: Applied a schedule template (power feature usage)
        SUM(CASE WHEN ACTIVITY_NAME = 'Scheduling.Template.ApplyModal.Applied' 
            THEN 1 ELSE 0 END)                        AS template_applied_count,

        -- Goal 4: Created an open shift request (team scheduling)
        SUM(CASE WHEN ACTIVITY_NAME = 'Scheduling.OpenShiftRequest.Created' 
            THEN 1 ELSE 0 END)                        AS open_shift_request_count,

        -- Goal 5: Used punch clock (multi-module adoption)
        SUM(CASE WHEN ACTIVITY_NAME = 'PunchClock.PunchedIn' 
            THEN 1 ELSE 0 END)                        AS punched_in_count

    FROM stg
    GROUP BY ORGANIZATION_ID, is_converted
),

-- Convert counts to goal completion flags
goals AS (
    SELECT
        ORGANIZATION_ID,
        is_converted,

        -- Goal completion flags (1 = completed, 0 = not completed)
        CASE WHEN shift_created_count >= 3 
            THEN 1 ELSE 0 END                         AS goal_1_shift_created,

        CASE WHEN shift_approved_count >= 1 
            THEN 1 ELSE 0 END                         AS goal_2_shift_approved,

        CASE WHEN template_applied_count >= 1 
            THEN 1 ELSE 0 END                         AS goal_3_template_applied,

        CASE WHEN open_shift_request_count >= 1 
            THEN 1 ELSE 0 END                         AS goal_4_open_shift_request,

        CASE WHEN punched_in_count >= 1 
            THEN 1 ELSE 0 END                         AS goal_5_punched_in,

        -- Raw counts for reference
        shift_created_count,
        shift_approved_count,
        template_applied_count,
        open_shift_request_count,
        punched_in_count

    FROM activity_counts
)

SELECT * FROM goals